In [28]:
import pandas as pd

# BEGIN: Load Data

# Load the JSON data into a DataFrame
df = pd.read_json('results/prioritized_data.json').T
# END: Load Data

In [36]:
# BEGIN: Sort by patch_score and assign new index
sorted_df = df.sort_values(by='priority_score', ascending=False).reset_index(drop=False, names='finding_link')
# END: Sort by patch_score and assign new index

In [37]:
sorted_df.head()

,finding_link,scrapping_date,vulnerabilities,impact,publication_date,client_source_link,authors,n_authors,contains_github_link,audit_content,...,poc_content_types,has_poc,has_mitigation_proposal,has_patch_reference,audit_quality,patch_score,priority_score,verification,correctness,category
0,https://solodit.cyfrin.io/issues/m-1-incorrect...,2025-10-03,[logic error],Medium,"Apr 4, 2024",https://app.sherlock.xyz/audits/contests/205,"lemonmon, obront, Stiglitz and 1 more",4,yes,Source: https://github.com/sherlock-audit/2024...,...,"[text, code]",yes,yes,yes,excellent,88,145.0,not_attempted,not_evaluated,patch_and_poc
1,https://solodit.cyfrin.io/issues/m-01-multical...,2025-10-03,[access control],Medium,"Jun 10, 2024",https://code4rena.com/reports/2024-06-size,"BoltzmannBrain, inzinko, Inspex and 29 more",32,yes,https://github.com/code-423n4/2024-06-size/blo...,...,"[text, code, link]",yes,yes,yes,excellent,82,143.0,not_attempted,not_evaluated,patch_and_poc
2,https://solodit.cyfrin.io/issues/h-01-maker-bu...,2025-10-04,[unchecked external calls],High,"Jun 14, 2022",https://code4rena.com/contests/2022-06-infinit...,"WatchPug, unforgiven, PwnedNoMore and 1 more",4,yes,"Submitted by WatchPug, also found by 0xsanson,...",...,[empty],yes,yes,yes,excellent,90,138.0,not_attempted,not_evaluated,patch_and_poc
3,https://solodit.cyfrin.io/issues/h-04-vaultmin...,2025-10-03,[access control],High,"Jul 7, 2023",https://code4rena.com/reports/2023-07-pooltoge...,"keccak123, 0xStalin, josephdara and 35 more",38,yes,The\nVault.mintYieldFee\nexternal function is ...,...,"[code, link, text]",yes,yes,yes,excellent,79,137.5,not_attempted,not_evaluated,patch_and_poc
4,https://solodit.cyfrin.io/issues/h-2-reentranc...,2025-10-04,"[reentrancy, unchecked external calls, flash l...",High,"Feb 16, 2024",https://app.sherlock.xyz/audits/contests/137,"zzykxx, 0xadrii",2,yes,Source: https://github.com/sherlock-audit/2023...,...,"[text, link]",yes,yes,yes,excellent,84,136.0,not_attempted,not_evaluated,patch_and_poc


In [38]:
sorted_df.to_csv('../strike-hive/results/final_order_dataset.csv', index=True, header=True, index_label='finding_id')

# Now I want some metrics on the vulnerabilities added on results.json

In [39]:
rawdf = pd.read_json('results/results.json').T
rawdf.head()

,scrapping_date,vulnerabilities,impact,publication_date,client_source_link,authors,n_authors,contains_github_link,audit_content,github_commit,github_pr,patch_link,fix_commit,mitigation_code,mitigation_content_types,ai_summary,poc_content_types
https://solodit.cyfrin.io/issues/missing-access-control-allows-nonce-manipulation-trailofbits-none-gemini-smart-wallet-pdf,2025-10-03,[access control],Low,"Aug 15, 2025",https://github.com/trailofbits/publications/bl...,"Coriolan Pinhas Trail of Bits PUBLIC, Anish Naik",2,no,Diﬃculty: Low\nType: Data Validation\nDescript...,False,False,False,False,False,[text],,[]
https://solodit.cyfrin.io/issues/h-01-lack-of-access-control-in-agentnftv2addvalidator-enables-unauthorized-validator-injection-and-causes-reward-accounting-inconsistencies-code4rena-virtuals-protocol-virtuals-protocol-git,2025-10-03,[access control],High,"Aug 4, 2025",https://code4rena.com/reports/2025-04-virtuals...,"testnate, Damboy, Olugbenga and 18 more",21,yes,<https://github.com/code-423n4/2025-04-virtual...,False,False,False,False,True,"[code, text]",,[]
https://solodit.cyfrin.io/issues/missing-access-control-in-sdlvestingstakereleasabletokens-cyfrin-none-stakelink-vesting-markdown,2025-10-03,[access control],Medium,"Aug 2, 2025",https://github.com/solodit/solodit_content/blo...,"InAllHonesty, Immeas",2,no,Description:\nThe\nSDLVesting::stakeReleasable...,False,False,False,True,False,[],,[]
https://solodit.cyfrin.io/issues/missing-access-control-on-critical-feecontroller-setters-cyfrin-none-octodefi-markdown,2025-10-03,[access control],High,"Jul 17, 2025",https://github.com/solodit/solodit_content/blo...,"Giovanni Di Siena, Farouk",2,no,Description:\nFeeController.setFunctionFeeConf...,False,True,False,False,False,[],,[]
https://solodit.cyfrin.io/issues/10-misaligned-access-control-on-settopnpools-function-code4rena-audit-507-audit-507-git,2025-10-03,[access control],Low,"Jul 3, 2025",https://code4rena.com/reports/2025-05-blackhole,,1,yes,<https://github.com/code-423n4/2025-05-blackho...,False,False,False,False,False,[text],,[]


In [1]:
rawdf

NameError: name 'rawdf' is not defined

In [5]:
# BEGIN: Vulnerability Analysis - POC and Patch Counts per Vulnerability Type

import pandas as pd
from collections import defaultdict

# Load the raw data
rawdf = pd.read_json('results/results.json').T

# Function to determine if a finding has a POC
def has_poc(row):
    """Check if a finding has a POC based on poc_content_types"""
    poc_types = row.get('poc_content_types', [])
    return len(poc_types) > 0 and poc_types != ['empty']

# Function to determine if a finding has a patch
def has_patch(row):
    """Check if a finding has a patch based on various patch-related fields"""
    patch_fields = ['patch_link', 'fix_commit', 'github_pr', 'github_commit', 'mitigation_code']
    return any(row.get(field, False) for field in patch_fields)

# Create analysis results
vulnerability_stats = defaultdict(lambda: {'total': 0, 'with_poc': 0, 'with_patch': 0})

# Process each finding
for idx, row in rawdf.iterrows():
    vulnerabilities = row.get('vulnerabilities', [])
    has_poc_flag = has_poc(row)
    has_patch_flag = has_patch(row)
    
    # Count for each vulnerability type in this finding
    for vuln_type in vulnerabilities:
        vulnerability_stats[vuln_type]['total'] += 1
        if has_poc_flag:
            vulnerability_stats[vuln_type]['with_poc'] += 1
        if has_patch_flag:
            vulnerability_stats[vuln_type]['with_patch'] += 1

# Convert to DataFrame for better visualization
results = []
for vuln_type, stats in vulnerability_stats.items():
    results.append({
        'Vulnerability_Type': vuln_type,
        'Total_Findings': stats['total'],
        'With_POC': stats['with_poc'],
        'With_Patch': stats['with_patch']
    })

# Create DataFrame and sort by total findings
vulnerability_analysis_df = pd.DataFrame(results)
vulnerability_analysis_df = vulnerability_analysis_df.sort_values('Total_Findings', ascending=False).reset_index(drop=True)

# Calculate totals
total_findings = vulnerability_analysis_df['Total_Findings'].sum()
total_pocs = vulnerability_analysis_df['With_POC'].sum()
total_patches = vulnerability_analysis_df['With_Patch'].sum()

# Add totals row
totals_row = pd.DataFrame({
    'Vulnerability_Type': ['TOTAL'],
    'Total_Findings': [total_findings],
    'With_POC': [total_pocs],
    'With_Patch': [total_patches]
})

# Combine main results with totals
final_df = pd.concat([vulnerability_analysis_df, totals_row], ignore_index=True)

# Display results
print("Vulnerability Analysis: POC and Patch Counts per Vulnerability Type")
print("=" * 70)
print(final_df.to_string(index=False))

# END: Vulnerability Analysis - POC and Patch Counts per Vulnerability Type

Vulnerability Analysis: POC and Patch Counts per Vulnerability Type
       Vulnerability_Type  Total_Findings  With_POC  With_Patch
           access control            1043       235         336
              logic error             770       224         399
               reentrancy             760       210         325
               flash loan             356       139         126
  denial of service (DoS)             309        82         137
 lack of input validation             276        58         131
 unchecked external calls             250       103         125
price oracle manipulation             230        77          75
         integer overflow             175        23          77
        integer underflow              79        18          32
      insecure randomness               8         1           5
                    TOTAL            4256      1170        1768


In [6]:
# BEGIN: Deduplicated Vulnerability Analysis - Each Finding Counted Once

# Count findings deduplicated (each finding counted only once)
total_unique_findings = len(rawdf)
unique_findings_with_poc = 0
unique_findings_with_patch = 0

# Process each finding once
for idx, row in rawdf.iterrows():
    has_poc_flag = has_poc(row)
    has_patch_flag = has_patch(row)
    
    if has_poc_flag:
        unique_findings_with_poc += 1
    if has_patch_flag:
        unique_findings_with_patch += 1

# Create deduplicated summary
deduplicated_summary = pd.DataFrame({
    'Metric': ['Total Unique Findings', 'Findings with POC', 'Findings with Patch'],
    'Count': [total_unique_findings, unique_findings_with_poc, unique_findings_with_patch]
})

# Display deduplicated results
print("Deduplicated Analysis: Each Finding Counted Once")
print("=" * 50)
print(deduplicated_summary.to_string(index=False))

# Calculate percentages
poc_percentage = (unique_findings_with_poc / total_unique_findings * 100) if total_unique_findings > 0 else 0
patch_percentage = (unique_findings_with_patch / total_unique_findings * 100) if total_unique_findings > 0 else 0

print(f"\nPercentages:")
print(f"Findings with POC: {poc_percentage:.2f}%")
print(f"Findings with Patch: {patch_percentage:.2f}%")

# END: Deduplicated Vulnerability Analysis - Each Finding Counted Once

Deduplicated Analysis: Each Finding Counted Once
               Metric  Count
Total Unique Findings   3814
    Findings with POC   1028
  Findings with Patch   1588

Percentages:
Findings with POC: 26.95%
Findings with Patch: 41.64%


In [4]:
vulnerability_analysis_df.to_latex('results/vulnerability_analysis.tex', index=False)